# mini-llm-runner: Building Generative LLM from Scratch
**Author:** Vikash Mehta ([@vkshdev](https://github.com/vkshdev))  
**Project:** build-mini-LLM 

---

In this notebook, Their is two-stage training pipeline :
1. **Phase 1 (Pre-training):** Training foundational model on `roneneldan/TinyStories` (~160M tokens) to learn grammar, vocabulary, and common-sense reasoning.
2. **Phase 2 (Conversational SFT):** Aligning the base weights into an interactive assistant using `tatsu-lab/alpaca` instructions with ChatML formatting and prompt loss masking.
3. **Telemetry & Verification:** Streaming loss curves to Weights & Biases (W&B), running inference benchmarks, and saving model checkpoints.

## 1. Checking Hardware & Runtime Environment
Before launching the training run, verify the Kaggle has allocated an active CUDA device and check its available VRAM.

In [ ]:
import os
import sys
import torch

print("=" * 60)
print("MY RUNTIME HARDWARE AUDIT")
print("=" * 60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"Allocated GPUs:  {gpu_count}")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  [Device {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute: {props.major}.{props.minor}")
else:
    print("[Alert] No GPU detected. Please enable GPU acceleration in Kaggle notebook settings.")

!nvidia-smi

## 2. Cloning Codebase & Setting Up Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/vkshdev/build-mini-LLM.git"
PROJECT_DIR = "/kaggle/working/build-mini-LLM"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# Clone or pull latest changes from my repository
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning my repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already exists locally. Pulling my latest commits...")
    !git -C {PROJECT_DIR} pull origin main

# Ensure my checkpoint output directory is ready
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Switch working directory and install my dependencies
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("[OK] Environment prepared and ready to train.")

## 3. Authenticating Weights & Biases Dashboard
To track loss convergence, learning rate schedule, and text generation samples over time.

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    my_wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=my_wandb_key)
    print("[OK] Successfully authenticated with W&B via Kaggle Secrets.")
except Exception:
    print("Kaggle secret not detected. Falling back to W&B login:")
    wandb.login()

## 4. Phase 1: Pre-training My Base 28M LLM on TinyStories
Train the core model across 160 million tokens from Microsoft Research's `TinyStories` dataset.

My training setup:
* **Sequence Length:** 256 tokens
* **Optimizer:** AdamW with linear warmup (250 steps) and cosine decay down to 10% peak LR
* **Precision:** FP16 Automatic Mixed Precision
* **Snapshots:** Checkpoints saved to `/kaggle/working/checkpoints` every 1,000 steps, with the best model logged to W&B


In [ ]:
!python -m src.train \
    --data_dir /kaggle/working/build-mini-LLM/data \
    --checkpoint_dir /kaggle/working/checkpoints \
    --max_steps 10000 \
    --batch_size 32 \
    --lr 6e-4 \
    --resume True

## 5. Phase 2: Conversational Fine-Tuning (Chat SFT)

In [ ]:
!python -m src.sft_train \
    --base_model /kaggle/working/checkpoints/best_pretrained_28m.pt \
    --output_dir /kaggle/working/checkpoints \
    --max_steps 1500 \
    --batch_size 16 \
    --lr 3e-5

## 6. Testing & Benchmarking My Chatbot


In [ ]:
import time
import torch
from src.config import ModelConfig
from src.tokenizer import get_tokenizer
from src.model import SmallLLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = get_tokenizer()
my_model = SmallLLM(ModelConfig()).to(device)

# Check for my fine-tuned model or fall back to base pre-trained
weights_path = "/kaggle/working/checkpoints/chat_model_28m.pt"
if not os.path.exists(weights_path):
    print("SFT weights not found. Loading my best pre-trained checkpoint instead.")
    weights_path = "/kaggle/working/checkpoints/best_pretrained_28m.pt"

print(f"Loading checkpoint from: {weights_path}")
ckpt = torch.load(weights_path, map_location=device)
state_dict = ckpt.get("model_state_dict", ckpt)
my_model.load_state_dict(state_dict)
my_model.eval()
print(f"[OK] Model successfully loaded on {device.upper()}.")

def ask_my_llm(user_query: str, max_new_tokens: int = 80, temperature: float = 0.7):
    prompt_str = f"<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"
    input_ids = torch.tensor(tokenizer.encode(prompt_str), dtype=torch.long, device=device).unsqueeze(0)
    
    t0 = time.time()
    output_ids = my_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=35,
        top_p=0.9,
        stop_id=tokenizer.im_end_id
    )
    dt = time.time() - t0
    
    full_text = tokenizer.decode(output_ids[0].tolist())
    response = tokenizer.extract_reply(full_text)
    num_generated = output_ids.size(1) - input_ids.size(1)
    throughput = num_generated / max(dt, 1e-4)
    
    print(f"User:        {user_query}")
    print(f"Assistant:   {response}")
    print(f"Performance: {num_generated} tokens generated in {dt:.2f}s ({throughput:.1f} tokens/sec)\n")

# Run test queries through my custom model
test_prompts = [
    "Hello! Who created you?",
    "Tell me a short story about an owl who was afraid of the dark.",
    "Why do birds sing in the morning?",
    "What is 5 plus 7?"
]

for q in test_prompts:
    ask_my_llm(q)

## 7. Reviewing Saved Checkpoints & Publishing

In [ ]:
import glob

print("MY CHECKPOINT ARTIFACT SUMMARY:")
print("=" * 60)
my_ckpts = glob.glob("/kaggle/working/checkpoints/*.pt")
if my_ckpts:
    for p in sorted(my_ckpts):
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  {os.path.basename(p):<30} | {size_mb:8.2f} MB")
else:
    print("No checkpoint files found yet.")
print("=" * 60)